# RSNA Knee Abnormality Detection — First Submission
**Team:** Gladiators (Utkarsh Milind Inamdar · Gaurav Hemant Lokhande)

In [ ]:
!pip install -q timm
print('Setup done.')

In [ ]:
import os, re, unicodedata, shutil
import numpy as np
import pandas as pd
import cv2
import pydicom
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import timm

print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# ── Debug: show what is mounted ────────────────────────────────────────────────
print('=== /kaggle/input contents ===')
for d in sorted(os.listdir('/kaggle/input')):
    full = f'/kaggle/input/{d}'
    try:
        files = os.listdir(full)[:5]
    except Exception:
        files = ['(unreadable)']
    print(f'  {d}/  ->  {files}')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
WORKING_DIR = '/kaggle/working'
CSV_DIR     = '/kaggle/input/rsna-knee-csv-data'

# Competition DICOMs mount under competitions/ when attached via API
DICOM_DIR = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
if not os.path.exists(DICOM_DIR):
    # Fallback: web-UI attachment mounts directly
    DICOM_DIR = '/kaggle/input/rsna-knee-abnormality-detection'

TRAIN_CSV    = f'{CSV_DIR}/train.csv'
TRAIN_SERIES = f'{CSV_DIR}/train_series.csv'
TEST_CSV     = f'{CSV_DIR}/test.csv'
TEST_SERIES  = f'{CSV_DIR}/test_series.csv'
SAMPLE_SUB   = f'{CSV_DIR}/sample_submission.csv'
PSEUDO_CSV   = f'{WORKING_DIR}/train_pseudo_labeled.csv'
MODEL_PATH   = f'{WORKING_DIR}/model_best.pt'
SUB_PATH     = f'{WORKING_DIR}/submission.csv'

PLANE         = 'Sagittal'
TARGET_SLICES = 16        # reduced from 24 to save GPU memory
IMG_SIZE      = 224
BACKBONE      = 'efficientnet_b0'
EPOCHS        = 10
BATCH_SIZE    = 2         # reduced from 8 — 2*16=32 slices per forward pass
LR            = 1e-4
SEED          = 42
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_WORKERS   = 0         # 0 = no subprocess, stable on Kaggle

LABEL_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Device  : {DEVICE}')
print(f'CSV_DIR : {CSV_DIR}  exists={os.path.exists(CSV_DIR)}')
print(f'DICOM   : {DICOM_DIR}  exists={os.path.exists(DICOM_DIR)}')

## Stage 1 — Pseudo-Labels

In [ ]:
# Use pre-built pseudo-labels from uploaded dataset (already generated locally)
shutil.copy(f'{CSV_DIR}/train_pseudo_labeled.csv', PSEUDO_CSV)
pseudo_df = pd.read_csv(PSEUDO_CSV)
print(f'Pseudo-labels loaded: {len(pseudo_df)} studies')
print('Label prevalence (% positive):')
print((pseudo_df[LABEL_COLS] == 1).mean().mul(100).round(1).to_string())

## Stage 2 — Dataset and Model

In [ ]:
def load_dicom_slice(filepath):
    dcm = pydicom.dcmread(filepath)
    px = dcm.pixel_array.astype(np.float32)
    px = px * getattr(dcm, 'RescaleSlope', 1.0) + getattr(dcm, 'RescaleIntercept', 0.0)
    d = px.max() - px.min()
    return np.zeros_like(px) if d == 0 else (px - px.min()) / d

def load_and_resize_series(series_dir, target_slices=16, target_size=(224, 224)):
    dcm_files = [os.path.join(series_dir, f) for f in os.listdir(series_dir) if f.endswith('.dcm')]
    if not dcm_files:
        raise ValueError(f'No DICOM files in: {series_dir}')
    def get_z(fp):
        try: return float(pydicom.dcmread(fp, stop_before_pixels=True).ImagePositionPatient[2])
        except: return float(os.path.splitext(os.path.basename(fp))[0])
    dcm_files.sort(key=get_z)
    volume = np.stack([cv2.resize(load_dicom_slice(f), target_size, interpolation=cv2.INTER_LINEAR) for f in dcm_files])
    if volume.shape[0] != target_slices:
        volume = ndimage.zoom(volume, (target_slices / volume.shape[0], 1, 1), order=1)
    return volume.astype(np.float32)


class RSNAKneeDataset(Dataset):
    def __init__(self, csv_file, series_csv, dicom_dir, plane='Sagittal',
                 target_slices=16, target_size=(224, 224), is_train=True):
        self.dicom_dir     = dicom_dir
        self.target_slices = target_slices
        self.target_size   = target_size
        self.is_train      = is_train
        self.label_cols    = LABEL_COLS

        self.labels_df  = pd.read_csv(csv_file)
        label_uids      = set(self.labels_df['StudyInstanceUID'].values)

        series_df       = pd.read_csv(series_csv)
        plane_series    = series_df[series_df['Anatomical_Plane'].str.lower() == plane.lower()]
        # *** Only keep studies that are in the labels CSV (respects train/val split) ***
        self.plane_series = plane_series[plane_series['StudyInstanceUID'].isin(label_uids)]
        self.studies      = self.plane_series['StudyInstanceUID'].unique()

    def __len__(self): return len(self.studies)

    def __getitem__(self, idx):
        study_uid  = self.studies[idx]
        series_uid = self.plane_series[self.plane_series['StudyInstanceUID'] == study_uid].iloc[0]['SeriesInstanceUID']
        subfolder  = 'train_series' if self.is_train else 'test_series'
        series_dir = os.path.join(self.dicom_dir, subfolder, study_uid, series_uid)
        try:
            volume = load_and_resize_series(series_dir, self.target_slices, self.target_size)
        except Exception:
            volume = np.zeros((self.target_slices, *self.target_size), dtype=np.float32)
        volume_tensor = torch.tensor(volume, dtype=torch.float32).unsqueeze(0)
        study_labels  = self.labels_df[self.labels_df['StudyInstanceUID'] == study_uid]
        if len(study_labels) > 0 and self.is_train:
            targets = study_labels.iloc[0][self.label_cols].values.astype(np.float32)
        else:
            targets = np.zeros(len(self.label_cols), dtype=np.float32)
        return volume_tensor, torch.tensor(targets, dtype=torch.float32)


class RSNAKneeModel(nn.Module):
    def __init__(self, backbone='efficientnet_b0', num_classes=12, pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(backbone, pretrained=pretrained, in_chans=1, num_classes=0, global_pool='avg')
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.encoder.num_features, num_classes))
    def forward(self, x):
        B, C, D, H, W = x.shape
        x = x.squeeze(1).reshape(B * D, 1, H, W)
        return self.head(self.encoder(x).reshape(B, D, -1).mean(dim=1))

print('Classes defined.')

## Stage 3 — Training (fp16 mixed precision)

In [ ]:
torch.manual_seed(SEED); np.random.seed(SEED)

df = pd.read_csv(PSEUDO_CSV)
df = df.dropna(subset=LABEL_COLS, how='all').reset_index(drop=True)
df[LABEL_COLS] = df[LABEL_COLS].fillna(0).astype(float)

train_ids, val_ids = train_test_split(df['StudyInstanceUID'].unique(), test_size=0.2, random_state=SEED)

os.makedirs('/kaggle/working/tmp', exist_ok=True)
df[df['StudyInstanceUID'].isin(train_ids)].to_csv('/kaggle/working/tmp/train_split.csv', index=False)
df[df['StudyInstanceUID'].isin(val_ids)].to_csv('/kaggle/working/tmp/val_split.csv',   index=False)

shared_kw = dict(series_csv=TRAIN_SERIES, dicom_dir=DICOM_DIR, plane=PLANE,
                 target_slices=TARGET_SLICES, target_size=(IMG_SIZE, IMG_SIZE), is_train=True)

train_ds = RSNAKneeDataset('/kaggle/working/tmp/train_split.csv', **shared_kw)
val_ds   = RSNAKneeDataset('/kaggle/working/tmp/val_split.csv',   **shared_kw)
print(f'Train: {len(train_ds)} studies | Val: {len(val_ds)} studies')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

device    = torch.device(DEVICE)
model     = RSNAKneeModel(backbone=BACKBONE, pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler()   # fp16 gradient scaler
best_val  = float('inf')

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train(); tl, ts = 0.0, 0
    for vols, tgts in tqdm(train_loader, desc=f'E{epoch:02d} train', leave=False):
        vols, tgts = vols.to(device), tgts.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(vols)
            mask   = ~tgts.isnan()
            loss   = criterion(logits[mask], tgts[mask])
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tl += loss.item(); ts += 1

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval(); vl, vs = 0.0, 0
    with torch.no_grad():
        for vols, tgts in tqdm(val_loader, desc=f'E{epoch:02d} val', leave=False):
            vols, tgts = vols.to(device), tgts.to(device)
            with autocast():
                logits = model(vols)
                mask   = ~tgts.isnan()
                loss   = criterion(logits[mask], tgts[mask])
            vl += loss.item(); vs += 1

    scheduler.step()
    tl /= max(ts,1); vl /= max(vs,1)
    print(f'Epoch {epoch:02d}/{EPOCHS} | train: {tl:.4f} | val: {vl:.4f}')
    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  -> best saved (val={vl:.4f})')

print(f'\nTraining done. Best val: {best_val:.4f}')

## Stage 4 — Inference & Submission

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

test_ds = RSNAKneeDataset(
    TEST_CSV, series_csv=TEST_SERIES, dicom_dir=DICOM_DIR,
    plane=PLANE, target_slices=TARGET_SLICES, target_size=(IMG_SIZE, IMG_SIZE), is_train=False
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f'Test studies: {len(test_ds)}')

all_preds = []
with torch.no_grad():
    for vols, _ in tqdm(test_loader, desc='inference'):
        with autocast():
            preds = torch.sigmoid(model(vols.to(device)))
        all_preds.append(preds.cpu().float().numpy())
all_preds = np.concatenate(all_preds)

series_df    = pd.read_csv(TEST_SERIES)
ordered_uids = series_df[series_df['Anatomical_Plane'].str.lower() == PLANE.lower()]['StudyInstanceUID'].unique()
submission   = pd.DataFrame(all_preds, columns=LABEL_COLS)
submission.insert(0, 'StudyInstanceUID', ordered_uids[:len(submission)])

sample_sub = pd.read_csv(SAMPLE_SUB)
submission = sample_sub[['StudyInstanceUID']].merge(submission, on='StudyInstanceUID', how='left')
submission[LABEL_COLS] = submission[LABEL_COLS].fillna(0.5)
submission.to_csv(SUB_PATH, index=False)
print(f'Saved: {SUB_PATH}  shape={submission.shape}')
submission.head()